# Minh họa năm cấu hình phân cụm sâu

AE + K-means, DEC, IDEC, DCN và IDEC + BRB. Dữ liệu Digits kèm sẵn. Code NumPy minh họa cơ chế; cấu hình khác với bài báo gốc.

Mở notebook sau khi giải nén và cài requirements. Có thể chạy bằng Jupyter trên máy hoặc tải toàn bộ ZIP lên Colab rồi giải nén. Trong Colab, giải nén vào `/content` trước khi chạy các ô bên dưới.

In [ ]:
from pathlib import Path
import sys
import subprocess
from datetime import datetime

# Tìm thư mục dự án từ vị trí notebook hoặc vị trí giải nén.
candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parents[1], Path("/content/Deep_Clustering_Demo")]
ROOT = next((p for p in candidates if (p / "run_demo.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Hãy giải nén thư mục Deep_Clustering_Demo rồi sửa ROOT cho đúng vị trí.")
print(ROOT)

## Cài thư viện nếu môi trường chưa có
Ô sau cần mạng để cài thư viện. Việc chạy thí nghiệm không cần mạng sau khi đã cài xong.

In [ ]:
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

bundle = np.load(ROOT / "data" / "digits_1797.npz", allow_pickle=False)
X, y = bundle["X"], bundle["y"]
print("X:", X.shape, "y:", y.shape, "range:", X.min(), X.max())
plt.imshow(X[0].reshape(8, 8), cmap="gray_r")
plt.title(f"Digit {y[0]} (nhãn chỉ dùng đánh giá)")
plt.axis("off");

## Xem lần chạy đã kiểm chứng hoặc chạy mới
Mặc định đọc kết quả 25 lượt đã chạy trên máy này. Đổi RUN_NEW_EXPERIMENT thành True để chạy mới. Mỗi seed huấn luyện lại autoencoder; các phương pháp cùng seed dùng chung điểm khởi đầu. Đổi SEEDS thành [0] nếu muốn thử nhanh.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
RUN_NEW_EXPERIMENT = False
if RUN_NEW_EXPERIMENT:
    OUT = ROOT / "results" / ("legacy_notebook_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
    command = [sys.executable, str(ROOT / "run_demo.py"), "--methods", "AE_KMeans", "DEC", "IDEC", "DCN", "IDEC_BRB", "--output", str(OUT), "--seeds", *map(str, SEEDS)]
    subprocess.check_call(command, cwd=ROOT)
    subprocess.check_call([sys.executable, str(ROOT / 'checks' / 'verify_experiment.py'), '--results', str(OUT)], cwd=ROOT)
    subprocess.check_call([sys.executable, str(ROOT / 'scripts/summarize_survey.py'), '--output', str(OUT / 'survey_counts.json')], cwd=ROOT)
else:
    OUT = ROOT / "archive/results_verified_20261012"
    assert (OUT / "summary.csv").exists(), "Chưa có kết quả: đổi RUN_NEW_EXPERIMENT thành True."
print(OUT)

In [ ]:
import pandas as pd
summary = pd.read_csv(OUT / "summary.csv")
summary

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(OUT / "performance.png")))
display(Image(filename=str(OUT / "stability.png")))
display(Image(filename=str(OUT / "loss_curves.png")))

## Xem cơ chế Q và P trên cùng dữ liệu
Q là soft assignment Student-t sau pretrain và K-means. P tăng trọng số assignment tự tin và hiệu chỉnh tần suất cụm, tính từ toàn bộ X. Mỗi hàng Q/P có tổng bằng 1; không dùng y. Trong DEC/IDEC, P được giữ cố định trong 5 epoch rồi cập nhật, nên loss có thể tạo răng cưa và thay đổi giá trị khi target đổi.

In [ ]:
sys.path.insert(0, str(ROOT))
from src.models import Autoencoder, assignments, target_distribution
import json
config = json.loads((OUT / 'config.json').read_text(encoding='utf-8'))
seed = config['seeds'][0]
model = Autoencoder(X.shape[1], config['latent_dim'], seed)
with np.load(OUT / 'models' / f'pretrain_seed{seed}.npz') as saved:
    model.params = {key: saved[key].copy() for key in model.params}
with np.load(OUT / 'models' / f'initialization_seed{seed}.npz') as saved:
    centers = saved['centers'].copy()
Q = assignments(model.encode(X), centers)
P = target_distribution(Q)
np.testing.assert_allclose(Q.sum(axis=1), 1)
np.testing.assert_allclose(P.sum(axis=1), 1)
display(pd.DataFrame(Q[:5]).round(3))
display(pd.DataFrame(P[:5]).round(3))

In [ ]:
import json
print(json.dumps(json.loads((OUT / 'verification.json').read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
display(pd.read_csv(ROOT / 'data' / 'paper_tracker.csv').fillna(''))

## Cách hiểu kết quả
ACC, NMI, ARI trong CSV nằm trên thang 0–1. Các biểu đồ hiệu năng đổi sang phần trăm. Pairwise ARI so sánh hai kết quả phân cụm trên cùng mẫu, không dùng nhãn thật. Năm seed tạo ra 10 cặp cho mỗi phương pháp; các cặp có dùng chung lần chạy nên không phải 10 quan sát độc lập.

Kết quả của bộ Digits nhỏ chưa xác định phương pháp tốt nhất trên MNIST hoặc toàn bộ deep clustering. Thư mục results chứa kết quả tham chiếu đã chạy kèm theo gói.